# Exercise - Assignment task-Find the best text classification model for the sentimental analysis (assignment submission)

The tasks for this part is use the grid search to:
1. Identify which vectorization method works the best or basically not much difference.
2. Identify which model, together with its corresponding hyperparameters, gives the best performance for traffic sentimental analysis.

You can either use the structure below or be a be a bit more explorative and try out other strategies we have discussed in the lecture/exercises to find the best parameters/model (e.g., Random Search, ROC curve,...).

In [11]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
%matplotlib inline
from sklearn.metrics import ConfusionMatrixDisplay as cmd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB
import os
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# The path of the dataset
url = 'https://raw.githubusercontent.com/zhenliangma/Applied-AI-in-Transportation/master/Exercise_4_Text_classification/Pakistani%20Traffic%20sentiment%20Analysis.csv'

# Load the data use the pandas
df = pd.read_csv(url)


#-*-*-*-*-*-*chose different vectorization-*-*-*-*-*-*

# Modification to run the 3 vectorization methods

#(1) CountVectorizer
vectorizerCV = CountVectorizer(ngram_range=(1, 2), stop_words='english',min_df=20)

#(2) #HashingVectorizer
vectorizerHV = HashingVectorizer(ngram_range=(1, 2), n_features=200)

#(3)TfidfVectorizer
vectorizerTV = TfidfVectorizer(
    min_df=20,
    norm='l2',
    smooth_idf=True,
    use_idf=True,
    ngram_range=(1, 1),
    stop_words='english'
    )

# Creates a dictionary with the parameters for the 3 vectorization methods
vectorizer={'CountVectorizer':vectorizerCV,'HashingVectorizer':vectorizerHV,'TfidfVectorizer':vectorizerTV}

#-*-*-*-*-*-*chose different vectorization-*-*-*-*-*-*

# split into train/test set
x = df['Text']
y = df['Sentiment']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

# Modification to save the 3 vectorization methods train and test data:

# apply the vectorizers
#x_train_vectorized = vectorizer.fit_transform(x_train)
#x_test_vectorized = vectorizer.transform(x_test)

# Creates a dictionary with the 3 vectorization transformed data
transformed_data = {}
# Creates an array to store the results and compare it
comparison_results = []

for name, vec in vectorizer.items():
    if name == 'HashingVectorizer':
        x_train_vectorized = vec.transform(x_train)
        x_test_vectorized = vec.transform(x_test)
    else:
        x_train_vectorized = vec.fit_transform(x_train)
        x_test_vectorized = vec.transform(x_test)

    print(f"Method: {name} ready to train model.")

# To use if the classification model is executed outside the for bucle
#    transformed_data[name] = (x_train_vec, x_test_vec)

#CHANGE THE VECTORIZATION METHOD NAME TO TRY
# Choose the Vectorization Method and transformed data to input on the model to compare
#x_train_vectorized, x_test_vectorized = transformed_data['TfidfVectorizer']

    # Instantiate a fixed baseline classifier (Logistic Regression)
    model = LogisticRegression(max_iter=1000, random_state=0)
    model.fit(x_train_vectorized, y_train)

    # Predict test samples and compute accuracy
    y_pred = model.predict(x_test_vectorized)
    acc = accuracy_score(y_test, y_pred) # Renamed variable from accuracy_score to acc
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Record the performance metrics
    comparison_results.append({
        'Vectorizer': name,
        'Accuracy Score': round(acc, 4), # Used acc here
        'Precision Score': round(prec, 4),
        'Recall Score': round(rec, 4),
        'F1 Score': round(f1, 4)
    })

# Convert summary list to a structured DataFrame and sort by performance
df_comparison = pd.DataFrame(comparison_results).sort_values(by='Accuracy Score', ascending=False)

# Display the final comparative table
print(df_comparison.to_string(index=False))




# here you can try use the grid search to find the best model parameter(a example is in SVM model)
#-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*

#(1)LR
# model = LogisticRegression(max_iter=1000, random_state=0)
# param_grid = {
#     'C': [0.001, 0.01, 0.1, 1, 10, 100],
# }

# model.fit(x_train_vectorized, y_train)
# print("Accuracy:", model.score(x_test_vectorized, y_test))


#(2)KNN
# model=KNeighborsClassifier()
# param_grid = {
#     'n_neighbors': [3, 5, 7, 9],
#     'weights': ['uniform', 'distance']
# }

#(3)RF
# model = RandomForestClassifier(random_state=0)
# param_grid = {
#     'n_estimators': [100, 200, 300],
#     'max_depth': [None, 10, 20, 30],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4]
# }

#(4)XGBoost
# model =  XGBClassifier()
# param_grid = {
#     'learning_rate': [0.01, 0.1, 0.2],
#     'n_estimators': [100, 200, 300],
#     'max_depth': [3, 4, 5]
# }


#(5)SVM
#model= SVC(probability=True)

# this is an example to use the grid search to find the best parameter for SVM model
# param_grid specifies the hyperparameter grid to search over： kernel types
# ('linear', 'rbf', 'poly') and regularization strength C values（0.1, 1, 10）.
#param_grid = {'kernel': ['linear', 'rbf', 'poly'],'C': [0.1, 1, 10]}

#`grid_search` performs a grid search with 5-fold cross-validation and evaluates models based on accuracy.
#grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy')

#`fit` method fits the model to the training data, systematically trying out all
# parameter combinations.
#grid_search.fit(x_train_vectorized, y_train)

#`best_params` and `best_score` store the best hyperparameters and their
# corresponding accuracy score.
#best_params = grid_search.best_params_
#print(best_params)
#best_score = grid_search.best_score_

#The `model` is updated with the best estimator found during the grid search,
# which can be used for further analysis.
#model = grid_search.best_estimator_

#(6)Naïve Bayes models
# model=BernoulliNB()
# param_grid = {'alpha': [0.1, 0.5, 1],'force_alpha': [True,False]}

#-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*


# cmd.from_estimator(
#     model,
#     x_test_vectorized,
#     y_test,
#     display_labels=['Positive','Negative'],
#     cmap='Blues',
#     xticks_rotation='vertical'
#     )

#calculate accuracy
#print('The accuracy of the model is: '+str(accuracy_score(y_test,model.predict(x_test_vectorized))))

Method: CountVectorizer ready to train model.
Method: HashingVectorizer ready to train model.
Method: TfidfVectorizer ready to train model.
       Vectorizer  Accuracy Score  Precision Score  Recall Score  F1 Score
  TfidfVectorizer          0.9621           0.9644        0.9644    0.9644
  CountVectorizer          0.9526           0.9812        0.9289    0.9543
HashingVectorizer          0.9479           0.9593        0.9422    0.9507


In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Assuming raw text data is already split:
# x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

# Define the dictionary of the three text vectorization techniques to compare
vectorizers = {
    # 1. CountVectorizer: Generates integer word occurrence counts based on an in-memory dictionary
    'CountVectorizer': CountVectorizer(
        ngram_range=(1, 2),
        stop_words='english',
        min_df=10
    ),

    # 2. TfidfVectorizer: Assigns statistical weights reflecting word importance (TF-IDF values between 0.0 and 1.0)
    'TfidfVectorizer': TfidfVectorizer(
        ngram_range=(1, 2),
        stop_words='english',
        min_df=10
    ),

    # 3. HashingVectorizer: Stateless and memory-efficient via hashing; does not store a vocabulary or accept min_df
    'HashingVectorizer': HashingVectorizer(
        ngram_range=(1, 2),
        stop_words='english',
        n_features=2**14,
        alternate_sign=False
    )
}

comparison_results = []

# Iterate over each vectorization technique
for name, vec in vectorizers.items():
    # Apply vectorization
    # HashingVectorizer is stateless and only uses transform; others require fit_transform on train data
    if name == 'HashingVectorizer':
        x_train_vec = vec.transform(x_train)
        x_test_vec = vec.transform(x_test)
        vocab_size = "N/A (Stateless)"
    else:
        x_train_vec = vec.fit_transform(x_train)
        x_test_vec = vec.transform(x_test)
        vocab_size = len(vec.vocabulary_)

    # Instantiate a fixed baseline classifier (Logistic Regression)
    clf = LogisticRegression(max_iter=1000, random_state=0)
    clf.fit(x_train_vec, y_train)

    # Predict test samples and compute accuracy
    y_pred = clf.predict(x_test_vec)
    acc = accuracy_score(y_test, y_pred)

    # Record the performance metrics
    comparison_results.append({
        'Vectorizer': name,
        'Vocabulary Size': vocab_size,
        'Test Accuracy': round(acc, 4)
    })

# Convert summary list to a structured DataFrame and sort by performance
df_comparison = pd.DataFrame(comparison_results).sort_values(by='Test Accuracy', ascending=False)

# Display the final comparative table
print(df_comparison.to_string(index=False))

       Vectorizer Vocabulary Size  Test Accuracy
HashingVectorizer N/A (Stateless)         0.9787
  TfidfVectorizer             305         0.9739
  CountVectorizer             305         0.9716


In [ ]:
# Here you change the reviews
text = 'Adayala road is clear'

# Make a prediction for this review
score=model.predict_proba(vectorizer.transform([text]))[0][1]

if score >0.5:
  attitude='negative'
else:
  attitude='positive'

print('The prediction result of this review is: '+ attitude)